Geospatial columns can look like ordinary binary data when their type information is lost between a database and a client. [GeoArrow](https://geoarrow.org/) avoids that by defining Arrow representations for geometry data, including Well-Known Binary (WKB).

This recipe shows what that looks like with [DuckDB's spatial extension](https://duckdb.org/docs/stable/core_extensions/spatial/overview) and ADBC. We will query a few `GEOMETRY` values, check the Arrow schema, and convert the WKB results to readable Well-Known Text (WKT).

## Setup

Install the Python dependencies. Importing `geoarrow.pyarrow` registers GeoArrow extension types with PyArrow so WKB columns are recognized when ADBC returns them:

In [1]:
%pip install -q dbc adbc-driver-manager pyarrow geoarrow-pyarrow


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Install the DuckDB ADBC driver:

In [2]:
!dbc install -q duckdb

Import PyArrow, GeoArrow, and the ADBC driver manager:

In [3]:
import geoarrow.pyarrow as ga
import pyarrow as pa
from adbc_driver_manager import dbapi

## Create Geospatial Data in DuckDB

Open an in-memory DuckDB database through ADBC:

In [4]:
connection = dbapi.connect(driver="duckdb")
cursor = connection.cursor()

Install and load DuckDB's spatial extension. `INSTALL` downloads the extension once; subsequent runs reuse the local copy:

In [5]:
cursor.execute("INSTALL spatial")
_ = cursor.execute("LOAD spatial")

Create three point geometries from longitude and latitude values and fetch the result directly as an Arrow table:

In [6]:
_ = cursor.execute(
    """
    SELECT
        city,
        ST_Point(longitude, latitude)::GEOMETRY AS geometry
    FROM (
        VALUES
            ('San Francisco', -122.4194, 37.7749),
            ('New York', -73.9857, 40.7484),
            ('London', -0.1276, 51.5072)
    ) places(city, longitude, latitude)
    """
)
places = cursor.fetch_arrow_table()
places

pyarrow.Table
city: string
geometry: extension<geoarrow.wkb<WkbType>>
----
city: [["San Francisco","New York","London"]]
geometry: [[010100000050FC1873D79A5EC0D0D556EC2FE34240,0101000000B3EA73B5157F52C0C7293A92CB5F4440,0101000000DA1B7C613255C0BFFE43FAEDEBC04940]]

## Inspect the GeoArrow WKB Type

The important part of the result is its schema. The `geometry` column is identified as GeoArrow WKB instead of being returned as an unlabelled binary column:

In [7]:
places.schema

city: string
geometry: extension<geoarrow.wkb<WkbType>>

Inspect both the logical extension name and its physical Arrow storage type:

In [8]:
geometry_type = places["geometry"].type
print(f"Extension name: {geometry_type.extension_name}")
print(f"Storage type: {geometry_type.storage_type}")

Extension name: geoarrow.wkb
Storage type: binary


The column has a logical type of `geoarrow.wkb` and uses Arrow `binary` as its physical storage. GeoArrow-aware tools can use the type metadata directly, while other WKB-compatible tools can still work with the underlying bytes.

## Convert WKB to Readable WKT

Use GeoArrow to convert the geometry column to WKT without writing a WKB parser:

In [9]:
readable_places = pa.table(
    {
        "city": places["city"],
        "geometry_wkt": ga.as_wkt(places["geometry"]),
    }
)
readable_places

pyarrow.Table
city: string
geometry_wkt: extension<geoarrow.wkt<WktType>>
----
city: [["San Francisco","New York","London"]]
geometry_wkt: [["POINT (-122.4194 37.7749)","POINT (-73.9857 40.7484)","POINT (-0.1276 51.5072)"]]

## Cleanup

In [10]:
cursor.close()
connection.close()